<a href="https://colab.research.google.com/github/robsongfk/SalesInsightPY/blob/main/salesinsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# =====================================================================
# RF01 - Criar ou Carregar o Dataset de Vendas (Código exato do PDF)
# =====================================================================
def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor", "Teclado", "Mouse", "Headset"]
    categorias = {
        "Notebook": "Computadores", "Smartphone": "Celulares",
        "Tablet": "Celulares", "Monitor": "Computadores",
        "Teclado": "Perifericos", "Mouse": "Perifericos", "Headset": "Perifericos"
    }
    precos = {
        "Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
        "Monitor": 1200, "Teclado": 250, "Mouse": 120, "Headset": 350
    }
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    data_inicio = datetime(2025, 1, 1)

    dados = []
    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza
        if random.random() < 0.05: quantidade = None # valor nulo
        if random.random() < 0.04: preco = None # valor nulo
        if random.random() < 0.06: produto = " " + produto + " " # espacos extras
        if random.random() < 0.03: data_txt = "DATA INVALIDA" # data invalida
        if random.random() < 0.10: # ruido no nome
            cliente = random.choice([
                cliente.upper().replace("_", "-"),
                cliente + "!!",
                " " + cliente,
                cliente.replace("Cliente_", "cliente#"),
            ])

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    return pd.DataFrame(dados)

# Gerar e salvar o CSV bruto
if not os.path.exists("vendas.csv"):
    df_bruto = gerar_dataset_vendas()
    df_bruto.to_csv("vendas.csv", index=False)
    print(f"Dataset gerado com {len(df_bruto)} registros.\n")


# =====================================================================
# RF02 - Inspecionar e Descrever os Dados
# =====================================================================
def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

# Testando as etapas 1 e 2
df_carregado = pd.read_csv("vendas.csv")
inspecionar_dados(df_carregado)

Dataset gerado com 200 registros.

=== INSPECAO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        10
preco_unitario     4
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente   produto     categoria        regiao  \
0         1  2025-05-21  cliente#016     Mouse   Perifericos       Sudeste   
1         2  2025-09-16  Cliente_039  Notebook  Computadores         Norte   
2         3  2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4  2025-

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14
...,...,...,...,...,...,...,...,...
195,196,2025-06-04,CLIENTE-038,Monitor,Computadores,Centro-Oeste,5.0,NaN
196,197,2025-09-15,Cliente_046,Tablet,Celulares,Sul,3.0,1748.92
197,198,2025-08-21,Cliente_048,Smartphone,Celulares,Sul,3.0,2185.60
198,199,2025-11-02,Cliente_006,Headset,Perifericos,Norte,4.0,368.94


In [6]:
import re
import numpy as np

# =====================================================================
# RF03 - Limpar e Tratar os Dados (datetime e regex)
# =====================================================================
def limpar_dados(df):
    """
    Limpa e trata o DataFrame de vendas.
    Retorna: (df_limpo, relatorio)
    """
    df_limpo = df.copy()
    qtd_inicial = len(df_limpo)

    # 1. Remover espaços extras nas colunas de texto
    colunas_texto = ['cliente', 'produto', 'categoria', 'regiao']
    for col in colunas_texto:
        df_limpo[col] = df_limpo[col].astype(str).str.strip()

    # 2. Converter data_venda e descartar datas inválidas (NaT)
    df_limpo['data_venda'] = pd.to_datetime(df_limpo['data_venda'], errors='coerce')
    qtd_com_datas = len(df_limpo.dropna(subset=['data_venda']))
    removidos_data = len(df_limpo) - qtd_com_datas
    df_limpo.dropna(subset=['data_venda'], inplace=True)

    # 3. Descartar nulos em quantidade e preco_unitario
    qtd_antes_nulos = len(df_limpo)
    df_limpo.dropna(subset=['quantidade', 'preco_unitario'], inplace=True)
    removidos_nulos = qtd_antes_nulos - len(df_limpo)

    # 4. Ajustar os tipos numéricos
    df_limpo['quantidade'] = df_limpo['quantidade'].astype(int)
    df_limpo['preco_unitario'] = df_limpo['preco_unitario'].astype(float)

    # 5. Padronizar o nome do cliente usando expressões regulares (regex)
    def tratar_cliente(c):
        # Mantém apenas letras, números e underscore
        c_limpo = re.sub(r"[^A-Za-z0-9_]", "", str(c).strip())
        # Extrai os números e força o padrão Cliente_NNN
        numeros = re.findall(r"\d+", c_limpo)
        if numeros:
            return f"Cliente_{int(numeros[0]):03d}"
        return c_limpo

    df_limpo['cliente'] = df_limpo['cliente'].apply(tratar_cliente)

    # 6. Montar o relatório de limpeza
    qtd_final = len(df_limpo)
    relatorio = {
        "Registros Iniciais": qtd_inicial,
        "Removidos (Data Inválida)": removidos_data,
        "Removidos (Valores Nulos)": removidos_nulos,
        "Registros Finais": qtd_final
    }

    print("\n=== RELATÓRIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"{chave}: {valor}")

    return df_limpo, relatorio

# =====================================================================
# RF04 - Criar Colunas Derivadas com Transformações Condicionais
# =====================================================================
def criar_colunas_derivadas(df):
    """
    Cria colunas derivadas no DataFrame limpo.
    """
    # Operação vetorizada para receita total
    df['receita_total'] = df['quantidade'] * df['preco_unitario']

    # Colunas temporais
    df['mes'] = df['data_venda'].dt.month

    # Dicionário de mapeamento para o nome do mês em português
    meses_pt = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
        5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
        9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    df['mes_nome'] = df['mes'].map(meses_pt)
    df['trimestre'] = "Q" + df['data_venda'].dt.quarter.astype(str)
    df['ano'] = df['data_venda'].dt.year

    # Transformação condicional com np.select
    condicoes = [
        df['receita_total'] < 500,
        (df['receita_total'] >= 500) & (df['receita_total'] < 5000),
        df['receita_total'] >= 5000
    ]
    faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]
    df['faixa_receita_item'] = np.select(condicoes, faixas, default="Nao Classificado")

    print("\n=== COLUNAS DERIVADAS CRIADAS ===")
    print("Colunas atuais:", list(df.columns))

    return df

# Testando as funções com o DataFrame que carregamos na etapa anterior
df_limpo, relatorio_limpeza = limpar_dados(df_carregado)
df_transformado = criar_colunas_derivadas(df_limpo)

# Visualizando os primeiros registros para conferir o resultado
display(df_transformado.head())


=== RELATÓRIO DE LIMPEZA ===
Registros Iniciais: 200
Removidos (Data Inválida): 4
Removidos (Valores Nulos): 13
Registros Finais: 183

=== COLUNAS DERIVADAS CRIADAS ===
Colunas atuais: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario', 'receita_total', 'mes', 'mes_nome', 'trimestre', 'ano', 'faixa_receita_item']


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario,receita_total,mes,mes_nome,trimestre,ano,faixa_receita_item
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90,205.80,5,Maio,Q2,2025,Baixo Valor
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1,1939.76,1939.76,3,Março,Q1,2025,Medio Valor
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6,3864.87,23189.22,11,Novembro,Q4,2025,Alto Valor
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10,2008.14,20081.40,7,Julho,Q3,2025,Alto Valor
5,6,2025-08-21,Cliente_041,Headset,Perifericos,Sudeste,2,337.41,674.82,8,Agosto,Q3,2025,Medio Valor


In [4]:
!rm vendas.csv